In [1]:
import random

import torch
import numpy as np
import matplotlib.pyplot as plt

import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset

In [2]:
def set_seed(seed=None, seed_torch=True):
  """
  Function that controls randomness. NumPy and random modules must be imported.

  Args:
    seed : Integer
      A non-negative integer that defines the random state. Default is `None`.
    seed_torch : Boolean
      If `True` sets the random seed for pytorch tensors, so pytorch module
      must be imported. Default is `True`.

  Returns:
    Nothing.
  """
  if seed is None:
    seed = np.random.choice(2 ** 32)
  random.seed(seed)
  np.random.seed(seed)
  if seed_torch:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

  print(f'Random seed {seed} has been set.')


# In case that `DataLoader` is used
def seed_worker(worker_id):
  """
  DataLoader will reseed workers following randomness in
  multi-process data loading algorithm.

  Args:
    worker_id: integer
      ID of subprocess to seed. 0 means that
      the data will be loaded in the main process
      Refer: https://pytorch.org/docs/stable/data.html#data-loading-randomness for more details

  Returns:
    Nothing
  """
  worker_seed = torch.initial_seed() % 2**32
  np.random.seed(worker_seed)
  random.seed(worker_seed)

In [3]:
def set_device():
  """
  Set the device. CUDA if available, CPU otherwise

  Args:
    None

  Returns:
    Nothing
  """
  device = "cuda" if torch.cuda.is_available() else "cpu"
  if device != "cuda":
    print("GPU is not enabled in this notebook. \n"
          "If you want to enable it, in the menu under `Runtime` -> \n"
          "`Hardware accelerator.` and select `GPU` from the dropdown menu")
  else:
    print("GPU is enabled in this notebook. \n"
          "If you want to disable it, in the menu under `Runtime` -> \n"
          "`Hardware accelerator.` and select `None` from the dropdown menu")

  return device


In [4]:
SEED = 2021
set_seed(seed=SEED)
DEVICE = set_device()

Random seed 2021 has been set.
GPU is not enabled in this notebook. 
If you want to enable it, in the menu under `Runtime` -> 
`Hardware accelerator.` and select `GPU` from the dropdown menu


In [ ]:
class Net(nn.Module):
    """
    Initialize MLP Network
    """

    def __init__(self, actv, input_feature_num, hidden_unit_nums, output_feature_num):
        """
        Initialize MLP Network parameters

        Args:
            actv: string
                Activation function
            input_feature_num: int
                Number of input features
            hidden_unit_nums: list
                Number of units per hidden layer, list of integers
            output_feature_num: int
                Number of output features

        Returns:
            Nothing
        """

        super(Net, self).__init__()
        self.input_feature_num = input_feature_num
        self.mlp = nn.Sequential()

        in_num = input_feature_num
        for i in np.arange(len(hidden_unit_nums)):
            out_num = hidden_unit_nums[i]
            layer = nn.Linear(in_features=in_num, out_features=out_num, bias=True)
            in_num = out_num
            self.mlp.add_module(f'Linear_{i}', layer)

            actv_layer = eval(f'nn.{actv}')
            self.mlp.add_module(f'Activation_{i}', actv_layer)

        out_layer = nn.Linear(in_features=in_num, out_features=output_feature_num, bias=True)
        self.mlp.add_module('Output_Linear', out_layer)


    def foward(self, x):
        """
        Simulate forward pass of MLP Network

        Args:
            x: torch.tensor
                Input data
        
        Returns:
            logits: Instance of MLP
                Forward pass of MLP
        """

        x = x.view(-1, self.input_feature_num)
        logits = self.mlp(x)
        
        return logits

In [ ]:
input = torch.zeros((100, 2))
net = Net(actv='LeakyReLU(0.1)', input_feature_num=2, hidden_unit_nums=[100, 10, 5], output_feature_num=1).to(DEVICE)
y = net(input.to(DEVICE))
print(f'The output shape is {y.shape} for an input of shape {input.shape}')